export CAC_API_KEY="API KEY GOES HERE" # Run this in terminal or Use .env file and add it to .gitignore

In [4]:
import pandas as pd
import requests
import os
# Environment variable for API key
# api_key = os.getenv("CAC_API_KEY")

# .env file
from dotenv import load_dotenv
load_dotenv()  # loads .env
api_key = os.getenv("CAC_API_KEY")

host = "http://a-colon.rz.uni-mannheim.de"
model = "mistral-nemo"


In [34]:
df = pd.read_csv("dataset/all_multi_paragraphs_2022_2026_translated.csv")

test_df = df.tail(5)
test_df

,article_id,pub,par_index,text_block,group,length,text_block_en
25348,BADZ-51308969944,BADZ,2,Von Daniela Weingärtner\n\nNachdem Frankreich ...,MIGR,118,By Daniela Weingärtner\n\nAfter France settled...
25349,BADZ-51308969944,BADZ,4,"Nun fordern die Regierungen von Italien, Griec...",MIGR,188,"Now the governments of Italy, Greece, Malta an..."
25350,BADZ-51308692203,BADZ,2,BZ: Ihr Film befasst sich mit der Arbeit der N...,MIGR,123,BZ: Your film deals with the work of the NGO S...
25351,BADZ-51309103090,BADZ,2,Von Franz Schmider\n\nDer Interregio 37 aus Zü...,MIGR,192,By Franz Schmider\n\nThe Interregio 37 from Zu...
25352,BADZ-51309103090,BADZ,3,Der Interregio 37 aus Zürich fährt pünktlich a...,MIGR,276,The Interregio 37 from Zurich arrives punctual...


### API TEST

In [ ]:
# Few-shot examples
example_instructions_expl = "Please classify each paragraph according to the given labels and explain why."
example_fewshot = {
    "input": [
        "This is the first example paragraph.",
        "This is the second example paragraph."
    ],
    "explanation": [
        "This paragraph talks about topic A, so label is X.",
        "This paragraph mentions topic B, so label is Y."
    ],
    "label": ["X", "Y"]
}
example_reminder_expl = "Always provide the reasoning before the final label."

# Function to construct few-shot prompt for each paragraph
def generate_prompt(paragraph):
    prompt = "Instructions: " + example_instructions_expl + "\n\n"
    for inp, expl, label in zip(example_fewshot["input"], example_fewshot["explanation"], example_fewshot["label"]):
        prompt += f"Example:\nInput: {inp}\nExplanation: {expl}\nLabel: {label}\n\n"
    prompt += example_reminder_expl + "\n\n"
    prompt += "Now classify or explain this paragraph:\n" + paragraph
    return prompt

# Loop through the dataset and call API
results = []

for idx, row in test_df.iterrows():
    paragraph = row['text_block']
    article_id = row['article_id']
    
    prompt = generate_prompt(paragraph)
    
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": "You are a helpful AI assistant."},
            {"role": "user", "content": prompt}
        ]
    }
    
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    
    url = f"{host}/v1/chat/completions"
    response = requests.post(url, json=payload, headers=headers)
    
    if response.status_code == 200:
        content = response.json()['choices'][0]['message']['content']
        results.append({
            "article_id": article_id,
            "par_index": row['par_index'],
            "text_block": paragraph,
            "model_output": content
        })
    else:
        print(f"Error with article {article_id}, paragraph {row['par_index']}. Status code: {response.status_code}")
        results.append({
            "article_id": article_id,
            "par_index": row['par_index'],
            "text_block": paragraph,
            "model_output": None
        })

# Convert results to DataFrame
results_df = pd.DataFrame(results)
print(results_df)

In [ ]:
import pandas as pd
import requests
import os

pd.head -3 dataset/all_multi_paragraphs_2022_2026.csv